### 1. Install DSSAT and Python Utilities

To run DSSAT, we typically need to compile the DSSAT Fortran engine or use pre-compiled binaries, and then configure the input files (`FILEX`, weather, soil). We can use `pip` to install helper libraries like `fao56` or wrapper libraries if available, but compiling the core engine is often required for the full crop simulation. Let's first install some necessary packages for handling agricultural data and look at how to run a basic simulation.

In [1]:
# Install libraries for handling spatial, weather, or crop model inputs
!pip install -q pandas numpy matplotlib

### 2. Conceptual Overview of DSSAT Management Inputs

In DSSAT, management events such as **planting**, **fertilization**, **irrigation**, **pesticide applications**, and **harvesting** are defined in the **`FILEX`** (Experimental file).

Here is an example structure of how management options and schedules are defined in a FILEX format, which the DSSAT Fortran engine (`DSCSM048.EXE`) reads to simulate the exact dates for harvesting or pesticide/irrigation events based on rules (e.g., soil moisture thresholds or maturity indices).

In [2]:
# Example of what a simplified DSSAT FILEX management section looks like:
filex_content = """
*EXP.DETAILS: CRCalendar Simulation

*PLANTING
@P PLATE PLPBY PLCOV PLDIW PLDEP PLPAR PLSPL PLSPO PLNAME
 1 81120   R     S     N    5.0   1.0   7.5   1.0

*HARVESTS
@H HDATE HSTG HCOM HSIZE HPC
 1 81280 GS001   H   -99   -99
"""
print("DSSAT uses structured text files like this to define crop schedules.")

DSSAT uses structured text files like this to define crop schedules.


### 3. Simulating Crop Stages (Growing Degree Days)

DSSAT predicts phenological phases (such as flowering, canopy closure, and maturity) using Growing Degree Days (GDD). We can simulate this concept in Python below to see how a model dynamically calculates the optimal harvest window based on daily temperature data.

In [3]:
import pandas as pd
import numpy as np

# Simulate 120 days of temperature weather data
np.random.seed(42)
dates = pd.date_range(start="2023-05-01", periods=120)
tmax = np.random.uniform(22, 34, size=120)
tmin = np.random.uniform(10, 20, size=120)

weather_df = pd.DataFrame({'Date': dates, 'Tmax': tmax, 'Tmin': tmin})

# Base temperature for a crop (e.g., Maize = 10°C)
T_base = 10

# Calculate daily GDD: ((Tmax + Tmin) / 2) - T_base
weather_df['Daily_GDD'] = ((weather_df['Tmax'] + weather_df['Tmin']) / 2) - T_base
weather_df['Daily_GDD'] = weather_df['Daily_GDD'].clip(lower=0) # GDD cannot be negative
weather_df['Cumulative_GDD'] = weather_df['Daily_GDD'].cumsum()

# Define Phenological Targets (Maturity at 1200 GDD)
maturity_gdd_target = 1200
harvest_row = weather_df[weather_df['Cumulative_GDD'] >= maturity_gdd_target].head(1)

if not harvest_row.empty:
    harvest_date = harvest_row['Date'].values[0]
    gdd_reached = harvest_row['Cumulative_GDD'].values[0]
    print(f"Predicted Harvest Date (Physiological Maturity reached at {maturity_gdd_target} GDD): {pd.to_datetime(harvest_date).strftime('%Y-%m-%d')}")
else:
    print("Crop did not reach maturity within the simulated timeframe.")

Predicted Harvest Date (Physiological Maturity reached at 1200 GDD): 2023-08-15


### 4. Advanced Decision Rule for Pesticide Application

Unlike harvesting, which heavily relies just on thermal heat accumulation (GDD), chemical application decisions must factor in wind, current rainfall (to prevent wash-off), humidity, and specific growth windows.

In [4]:
# Add simulated daily rainfall (mm) and relative humidity (%) to our weather data
np.random.seed(42)
weather_df['Rainfall'] = np.random.exponential(scale=2, size=120)  # Simulated rain in mm
weather_df['Humidity'] = np.random.uniform(40, 95, size=120)      # Relative humidity %

# Define a pesticide spraying rule:
# 1. Crop stage is vulnerable: Cumulative GDD is between 400 and 800 (early-to-mid vegetative stage)
# 2. Disease risk is high: Daily temperature is warm (avg > 20°C) and Relative Humidity > 80% for spore germination
# 3. Weather is favorable for application: No heavy rain forecasted (Rainfall < 2mm on that day to avoid wash-off)

weather_df['Avg_Temp'] = (weather_df['Tmax'] + weather_df['Tmin']) / 2

weather_df['Spray_Triggered'] = (
    (weather_df['Cumulative_GDD'] >= 400) & (weather_df['Cumulative_GDD'] <= 800) &
    (weather_df['Avg_Temp'] > 20) &
    (weather_df['Humidity'] > 80) &
    (weather_df['Rainfall'] < 2.0)
)

# Find the dates that meet these complex pesticide application conditions
spray_dates = weather_df[weather_df['Spray_Triggered']]

print("=== Predicted Pesticide Application Windows ===")
if not spray_dates.empty:
    for index, row in spray_dates.iterrows():
        print(f"Date: {row['Date'].strftime('%Y-%m-%d')} | GDD: {row['Cumulative_GDD']:.1f} | Temp: {row['Avg_Temp']:.1f}°C | Humidity: {row['Humidity']:.1f}% | Rain: {row['Rainfall']:.1f}mm")
else:
    print("No optimal spray window detected based on current constraints.")

=== Predicted Pesticide Application Windows ===
Date: 2023-06-07 | GDD: 421.4 | Temp: 20.4°C | Humidity: 81.9% | Rain: 0.2mm
Date: 2023-06-09 | GDD: 445.0 | Temp: 22.3°C | Humidity: 80.1% | Rain: 1.2mm
Date: 2023-06-28 | GDD: 658.3 | Temp: 21.0°C | Humidity: 91.5% | Rain: 0.1mm
Date: 2023-07-03 | GDD: 713.3 | Temp: 22.5°C | Humidity: 88.3% | Rain: 0.9mm
Date: 2023-07-06 | GDD: 745.8 | Temp: 20.9°C | Humidity: 84.9% | Rain: 0.3mm


### 5. Fetching Real Weather Data via Open-Meteo API

Instead of simulating synthetic weather data, we can call the Open-Meteo API. Let's write a cell to pull actual historical daily weather (temperature, relative humidity, and rainfall) for a specific farming location (e.g., latitude 42.03, longitude -93.62) and run our crop model dynamically on real data.

In [5]:
import requests
import pandas as pd

# Coordinates for Ames, Iowa (a major agricultural research location)
latitude = 42.03
longitude = -93.62
start_date = "2023-05-01"
end_date = "2023-08-31"

# API URL to fetch daily max/min temperatures, precipitation, and average relative humidity
url = f"https://archive-api.open-meteo.com/v1/archive?latitude={latitude}&longitude={longitude}&start_date={start_date}&end_date={end_date}&daily=temperature_2m_max,temperature_2m_min,precipitation_sum,relative_humidity_2m_mean&timezone=auto"

response = requests.get(url)
if response.status_code == 200:
    data = response.json()
    daily_data = data['daily']

    # Load into a pandas DataFrame
    real_weather_df = pd.DataFrame({
        'Date': pd.to_datetime(daily_data['time']),
        'Tmax': daily_data['temperature_2m_max'],
        'Tmin': daily_data['temperature_2m_min'],
        'Rainfall': daily_data['precipitation_sum'],
        'Humidity': daily_data['relative_humidity_2m_mean']
    })

    print(f"Successfully fetched {len(real_weather_df)} days of real weather data!")
    display(real_weather_df.head())
else:
    print(f"Failed to fetch data. Status code: {response.status_code}")

Successfully fetched 123 days of real weather data!


,Date,Tmax,Tmin,Rainfall,Humidity
0,2023-05-01,14.9,5.3,0.0,37
1,2023-05-02,15.8,4.6,0.0,34
2,2023-05-03,19.5,2.4,0.0,39
3,2023-05-04,25.9,7.2,0.0,35
4,2023-05-05,22.2,10.4,8.1,71


### 6. Run Calendar Predictions on Real Weather Data

Now, we apply our crop model logic directly on the real retrieved data.

In [6]:
# Calculate metrics on real data
T_base = 10
real_weather_df['Daily_GDD'] = (((real_weather_df['Tmax'] + real_weather_df['Tmin']) / 2) - T_base).clip(lower=0)
real_weather_df['Cumulative_GDD'] = real_weather_df['Daily_GDD'].cumsum()

# Predict Harvest
harvest_row_real = real_weather_df[real_weather_df['Cumulative_GDD'] >= 1200].head(1)
if not harvest_row_real.empty:
    print(f"[HARVEST] Predicted Harvest Date based on real 2023 weather: {harvest_row_real['Date'].dt.strftime('%Y-%m-%d').values[0]} (Cumulative GDD: {harvest_row_real['Cumulative_GDD'].values[0]:.1f})")
else:
    print("[HARVEST] The crop did not reach the required heat units to mature in this timeframe.")

# Predict Pesticide Window
real_weather_df['Avg_Temp'] = (real_weather_df['Tmax'] + real_weather_df['Tmin']) / 2
real_weather_df['Spray_Triggered'] = (
    (real_weather_df['Cumulative_GDD'] >= 400) & (real_weather_df['Cumulative_GDD'] <= 800) &
    (real_weather_df['Avg_Temp'] > 20) &
    (real_weather_df['Humidity'] > 80) &
    (real_weather_df['Rainfall'] < 2.0)
)

real_spray_dates = real_weather_df[real_weather_df['Spray_Triggered']]
print("\n=== Real Pesticide Application Windows (First 5 matches) ===")
if not real_spray_dates.empty:
    for index, row in real_spray_dates.head(5).iterrows():
        print(f"Date: {row['Date'].strftime('%Y-%m-%d')} | GDD: {row['Cumulative_GDD']:.1f} | Temp: {row['Avg_Temp']:.1f}°C | Humidity: {row['Humidity']:.1f}% | Rain: {row['Rainfall']:.1f}mm")
else:
    print("No pesticide window triggered based on the target constraints.")

[HARVEST] Predicted Harvest Date based on real 2023 weather: 2023-08-12 (Cumulative GDD: 1203.0)

=== Real Pesticide Application Windows (First 5 matches) ===
No pesticide window triggered based on the target constraints.


### 7. Building a Crop Calendar Web API with FastAPI

To allow other systems or mobile apps to query this model, we can build an API endpoint. Let's first install `fastapi`, `uvicorn`, and `nest-asyncio` (required to run web servers inside Colab notebooks).

In [7]:
!pip install -q fastapi uvicorn nest-asyncio

### 8. Define the API Web Server

Now, we define the web application. We'll set up a `/predict_calendar` endpoint that accepts any coordinates, start date, and parameters dynamically.

In [8]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import requests
import pandas as pd
import nest_asyncio
import uvicorn
from threading import Thread

# Enable running Uvicorn inside the Jupyter Notebook
nest_asyncio.apply()

app = FastAPI(title="Crop Calendar API", description="API to dynamically compute harvest and pesticide schedules.")

# Define the input data schema expected by the API
class CalendarRequest(BaseModel):
    latitude: float
    longitude: float
    start_date: str
    end_date: str
    t_base: float = 10.0
    maturity_gdd_target: float = 1200.0

@app.post("/predict_calendar")
def predict_calendar(req: CalendarRequest):
    # 1. Fetch real-time weather from Open-Meteo
    url = f"https://archive-api.open-meteo.com/v1/archive?latitude={req.latitude}&longitude={req.longitude}&start_date={req.start_date}&end_date={req.end_date}&daily=temperature_2m_max,temperature_2m_min,precipitation_sum,relative_humidity_2m_mean&timezone=auto"
    response = requests.get(url)

    if response.status_code != 200:
        raise HTTPException(status_code=400, detail="Failed to fetch weather data for the specified coordinates.")

    daily_data = response.json().get('daily', {})
    if not daily_data or 'time' not in daily_data:
         raise HTTPException(status_code=400, detail="No weather data found for this range.")

    df = pd.DataFrame({
        'Date': pd.to_datetime(daily_data['time']),
        'Tmax': daily_data['temperature_2m_max'],
        'Tmin': daily_data['temperature_2m_min'],
        'Rainfall': daily_data['precipitation_sum'],
        'Humidity': daily_data['relative_humidity_2m_mean']
    })

    # 2. Compute GDD
    df['Daily_GDD'] = (((df['Tmax'] + df['Tmin']) / 2) - req.t_base).clip(lower=0)
    df['Cumulative_GDD'] = df['Daily_GDD'].cumsum()
    df['Avg_Temp'] = (df['Tmax'] + df['Tmin']) / 2

    # 3. Predict Harvest Date
    harvest_row = df[df['Cumulative_GDD'] >= req.maturity_gdd_target].head(1)
    predicted_harvest = None
    if not harvest_row.empty:
        predicted_harvest = harvest_row['Date'].dt.strftime('%Y-%m-%d').values[0]

    # 4. Predict Pesticide Spray Dates
    df['Spray'] = (
        (df['Cumulative_GDD'] >= 400) & (df['Cumulative_GDD'] <= 800) &
        (df['Avg_Temp'] > 20) & (df['Humidity'] > 80) & (df['Rainfall'] < 2.0)
    )
    spray_dates = df[df['Spray']]['Date'].dt.strftime('%Y-%m-%d').tolist()

    return {
        "location": {"latitude": req.latitude, "longitude": req.longitude},
        "parameters": {"t_base": req.t_base, "maturity_gdd_target": req.maturity_gdd_target},
        "predicted_harvest_date": predicted_harvest,
        "optimal_pesticide_spray_dates": spray_dates
    }

# Helper to run FastAPI in a background thread so the notebook stays interactive
def run_api():
    uvicorn.run(app, host="127.0.0.1", port=8000)

server_thread = Thread(target=run_api, daemon=True)
server_thread.start()
print("FastAPI server is running locally on http://127.0.0.1:8000")

FastAPI server is running locally on http://127.0.0.1:8000


### 9. Test the API

We can test our active local API directly by executing a `POST` request to local endpoint `http://127.0.0.1:8000/predict_calendar`.

In [9]:
import time
time.sleep(2) # Give the server thread a moment to start up

payload = {
    "latitude": 42.03,
    "longitude": -93.62,
    "start_date": "2023-05-01",
    "end_date": "2023-08-31",
    "t_base": 10,
    "maturity_gdd_target": 1200
}

api_response = requests.post("http://127.0.0.1:8000/predict_calendar", json=payload)
print("API Status Code:", api_response.status_code)
print("API Response Body:")
import json
print(json.dumps(api_response.json(), indent=2))

INFO:     127.0.0.1:38034 - "POST /predict_calendar HTTP/1.1" 200 OK
API Status Code: 200
API Response Body:
{
  "location": {
    "latitude": 42.03,
    "longitude": -93.62
  },
  "parameters": {
    "t_base": 10.0,
    "maturity_gdd_target": 1200.0
  },
  "predicted_harvest_date": "2023-08-12",
  "optimal_pesticide_spray_dates": []
}
